# Human Action Classifier
**Schuldt et al. 2004. Recognizing Human Actions: A Local SVM Approach**

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import re

from pathlib import Path
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from joblib import dump

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


### 1. Data Loading

In [ ]:
# --- [수정] 1. 레이블 및 데이터 설정 ---
# ACTIONS에 empty를 반드시 추가 (7개 클래스)
ACTIONS = ["walking", "jogging", "running", "boxing", "handclapping", "handwaving", "empty"]
action_to_id = {a: i for i, a in enumerate(ACTIONS)}

# 공식 데이터셋 분할 기준
TRAIN_SUBJECTS = [11, 12, 13, 14, 15, 16, 17, 18]
VAL_SUBJECTS = [19, 20, 21, 23, 24, 25, 1, 4]
TEST_SUBJECTS = [22, 2, 3, 5, 6, 7, 8, 9, 10]

# --- [추가] 00sequences.txt 파서 ---
def get_segments_map(txt_path):
    segments_map = {}
    with open(txt_path, 'r') as f:
        for line in f:
            if 'frames' not in line: continue
            line = line.strip()
            parts = re.split(r'\s+', line)
            v_name = parts[0]
            # 프레임 숫자들만 추출하여 구간 생성
            nums = re.findall(r'\d+', line.split('frames')[-1])
            valid_ranges = []
            for i in range(0, len(nums), 2):
                if i + 1 < len(nums):
                    valid_ranges.append((int(nums[i]), int(nums[i+1])))
            
            if valid_ranges:
                # 구간 사이의 빈 틈을 empty_ranges로 계산 
                empty_ranges = []
                for i in range(len(valid_ranges) - 1):
                    e_start, e_end = valid_ranges[i][1] + 1, valid_ranges[i+1][0] - 1
                    if e_start < e_end:
                        empty_ranges.append((e_start, e_end))
                segments_map[v_name] = {'action': valid_ranges, 'empty': empty_ranges}
    return segments_map

# --- [수정] 구간별 비디오 로딩 함수 ---
def load_video_segment(path, start_f, end_f, resize=(160, 120)):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_f - 1) # 시작 프레임으로 이동
    frames = []
    curr_f = start_f
    while curr_f <= end_f:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames.append(cv2.resize(frame, resize).astype(np.float32) / 255.0)
        curr_f += 1
    cap.release()
    return torch.from_numpy(np.stack(frames)) if len(frames) >= 5 else None

# --- [수정] 데이터셋 빌드 로직 (Vocabulary용/Classifier용 통합) ---
def build_segmented_data(root, segments_map, subjects_list, kmeans=None, K=400, is_train=True):
    X_out, y_out, empty_pool = [], [], []
    all_vids = list(Path(root).glob("**/*.avi"))
    
    for v_path in all_vids:
        # 파일명에서 personID와 action 추출
        match = re.search(r"person(?P<person>\d+)_(?P<action>[a-z]+)_", v_path.name)
        if not match or int(match.group("person")) not in subjects_list: continue
        
        v_key = v_path.stem.replace('_uncomp', '')
        if v_key not in segments_map: continue
        
        # 1. Action 구간 처리
        for start, end in segments_map[v_key]['action']:
            video = load_video_segment(v_path, start, end)
            if video is not None:
                desc = extract_descriptors(video.to(device))
                if desc.shape[0] > 0:
                    if kmeans is None: X_out.append(desc.cpu()) # Vocab 학습용
                    else: # 히스토그램 생성용
                        labels = kmeans.predict(desc.cpu().numpy())
                        hist = np.bincount(labels, minlength=K).astype(np.float32)
                        X_out.append(torch.from_numpy(hist / (hist.sum() + 1e-8)))
                        y_out.append(action_to_id[match.group("action")])

        # 2. Empty 구간 처리
        for start, end in segments_map[v_key]['empty']:
            video = load_video_segment(v_path, start, end)
            if video is not None:
                desc = extract_descriptors(video.to(device))
                if desc.shape[0] > 0:
                    if kmeans is None: empty_pool.append(desc.cpu())
                    else:
                        labels = kmeans.predict(desc.cpu().numpy())
                        hist = np.bincount(labels, minlength=K).astype(np.float32)
                        empty_pool.append((torch.from_numpy(hist / (hist.sum() + 1e-8)), action_to_id["empty"]))

    # 훈련 시 empty 데이터 과잉 방지 (언더샘플링)
    if is_train and len(empty_pool) > 200:
        random.shuffle(empty_pool)
        empty_pool = empty_pool[:200]
    
    if kmeans is None: return X_out + empty_pool
    else:
        for h, l in empty_pool: X_out.append(h); y_out.append(l)
        return torch.stack(X_out), torch.tensor(y_out)

def encode_video_from_desc(desc, kmeans, K):
    labels = kmeans.predict(desc.cpu().numpy())
    hist = np.bincount(labels, minlength=K).astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return torch.from_numpy(hist)

### 2. Data Processing
#### 2.1. Transform Video To Image Sequence

#### 2.2. Construct Gaussian Scale Space
$$L(· , \sigma^2, \tau^2) = f \times g(· , \sigma^2, \tau^2)$$

In [ ]:
def gaussian_1d(kernel_size, sigma):
    x = torch.arange(kernel_size) - kernel_size // 2
    g = torch.exp(-(x ** 2) / (2 * (sigma ** 2)))
    return g / g.sum()

def gaussian_blur_3d(video, sigma, tau):
    # spatial blur
    g_xy = gaussian_1d(7, sigma)
    g_xy = g_xy[None, None, :, None] * g_xy[None, None, None, :]
    video = F.conv2d(video.unsqueeze(1), g_xy, padding=3).squeeze(1)

    # temporal blur
    g_t = gaussian_1d(7, tau)[None, None, :, None, None]
    video = F.conv3d(video.unsqueeze(0), g_t, padding=(3,0,0)).squeeze(0)

    return video

#### 2.3. Compute Second-moment Matrix From Spatio-temporal Gradients

$$\nabla L = (L_x, L_y, L_t)^T$$
$$\mu(·; \sigma^2, \tau^2) = g(·; s\sigma^2, s\tau^2) \times (\nabla L(\nabla L)^T)$$

In [ ]:
def gradients_3d(L):
	Lx = L[:, :, 2:] - L[:, :, :-2]
	Ly = L[:, 2:, :] - L[:, :-2, :]
	Lt = L[2:, :, :] - L[:-2, :, :]

	T = min(Lx.shape[0], Ly.shape[0], Lt.shape[0])
	H = min(Lx.shape[1], Ly.shape[1], Lt.shape[1])
	W = min(Lx.shape[2], Ly.shape[2], Lt.shape[2])

	return Lx[:T, :H, :W], Ly[:T, :H, :W], Lt[:T, :H, :W]

def second_moment_matrix(Lx, Ly, Lt, sigma=2.0, tau=1.5):
    J_xx = Lx * Lx
    J_xy = Lx * Ly
    J_xt = Lx * Lt
    J_yy = Ly * Ly
    J_yt = Ly * Lt
    J_tt = Lt * Lt

    def smooth(x):
        return gaussian_blur_3d(x, sigma, tau)

    return smooth(J_xx), smooth(J_xy), smooth(J_xt), smooth(J_yy), smooth(J_yt), smooth(J_tt)

#### 2.4. Detect Interest Point Using Harris Response

$$H = det(\mu) - k * trace^3(\mu)$$

In [ ]:
def harris_response(J, k=0.005):
    J_xx, J_xy, J_xt, J_yy, J_yt, J_tt = J

    det = (
        J_xx * (J_yy * J_tt - J_yt ** 2)
        - J_xy * (J_xy * J_tt - J_xt * J_yt)
        + J_xt * (J_xy * J_yt - J_xt * J_yy)
    )

    trace = J_xx + J_yy + J_tt
    return det - k * trace ** 3

def detect_interest_points(H, threshold_ratio=0.01):
    threshold = threshold_ratio * H.max()
    points = torch.nonzero(H > threshold)
    return points

#### 2.5. Extract Spatio-temporal Descriptors

$$\bold{l} = (L_x, L_y, L_t, L_{xx}, ..., L_{tttt}$$

In [ ]:
def extract_jet(L, x, y, t, sigma, tau):
    # first order
    Lx = L[t, y, x+1] - L[t, y, x-1]
    Ly = L[t, y+1, x] - L[t, y-1, x]
    Lt = L[t+1, y, x] - L[t-1, y, x]

    # second order
    Lxx = L[t, y, x+1] - 2*L[t, y, x] + L[t, y, x-1]
    Lyy = L[t, y+1, x] - 2*L[t, y, x] + L[t, y-1, x]
    Ltt = L[t+1, y, x] - 2*L[t, y, x] + L[t-1, y, x]

    jet = torch.tensor([
        sigma * Lx, sigma * Ly, tau * Lt,
        sigma**2 * Lxx, sigma**2 * Lyy, tau**2 * Ltt
    ], device=L.device)

    return jet

def extract_descriptors(video, sigma=1.5, tau=1.0):
    L = gaussian_blur_3d(video, sigma, tau)
    Lx, Ly, Lt = gradients_3d(L)
    J = second_moment_matrix(Lx, Ly, Lt, 2*sigma, 2*tau)
    H = harris_response(J)

    points = detect_interest_points(H)
    desc = []

    for t, y, x in points:
        if (
            t > 1 and y > 1 and x > 1 and
            t < L.shape[0]-2 and
            y < L.shape[1]-2 and
            x < L.shape[2]-2
        ):
            desc.append(extract_jet(L, x, y, t, sigma, tau))

    if len(desc) == 0:
        return torch.empty((0, 6), device=device)

    return torch.stack(desc)

#### 2.6. Build Visual Vocabulary

In [ ]:
segments_map = get_segments_map("00sequences.txt")
# raw descriptors 추출 (기존 build_vocabulary 대체)
raw_descriptors = build_segmented_data("Data", segments_map, TRAIN_SUBJECTS, kmeans=None)
X_vocab = torch.cat(raw_descriptors).numpy()
# K-means 학습 [cite: 13]
kmeans = MiniBatchKMeans(n_clusters=400, batch_size=4096, random_state=0).fit(X_vocab)

Learning vocabulary...


[mpeg4 @ 0x311d45880] ac-tex damaged at 8 6
[mpeg4 @ 0x311d45880] Error at MB: 74


In [ ]:
# 훈련 및 테스트 데이터셋을 구간 단위로 생성
X_train, y_train = build_segmented_data("Data", segments_map, TRAIN_SUBJECTS, kmeans=kmeans, K=400)
X_test, y_test = build_segmented_data("Data", segments_map, TEST_SUBJECTS, kmeans=kmeans, K=400, is_train=False)

In [ ]:
# 기존 train_svm 및 evaluate 함수 그대로 사용 [cite: 18, 19]
clf, scaler = train_svm(X_train, y_train)
acc, cm = evaluate(clf, scaler, X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")
print("Confusion Matrix:\n", cm)

#### 2.7. Encode Video as Histogram

In [ ]:
def encode_video(video, kmeans, K):
    desc = extract_descriptors(video)

    if desc.numel() == 0:
        return torch.zeros(K)

    labels = kmeans.predict(desc.cpu().numpy())
    hist = np.bincount(labels, minlength=K).astype(np.float32)
    hist /= (hist.sum() + 1e-8)

    return torch.from_numpy(hist)

In [ ]:
save_dir = Path("artifact")
torch.save({
	"k": 400,
	"actions": ["walking", "jogging", "running", "boxing", "handclapping", "handwaving", "empty"],
	"X_train": X_train, "y_train": y_train,
	"X_val": X_val, "y_val": y_val,
	"X_test": X_test, "y_test": y_test,
}, save_dir / "kth_features.pt")

### 3. Train Classifier

In [ ]:
def train_svm(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = LinearSVC(C=1.0, max_iter=5000)
    clf.fit(Xs, y.numpy())

    return clf, scaler

In [ ]:
print("Training classifier...")
clf, scaler = train_svm(X_train, y_train)

Training classifier...


### 4. Model Evaluation

In [ ]:
def evaluate(clf, scaler, X, y):
    Xs = scaler.transform(X.numpy())
    y_pred = clf.predict(Xs)

    acc = accuracy_score(y.numpy(), y_pred)
    cm = confusion_matrix(y.numpy(), y_pred)

    return acc, cm

In [ ]:
print("---- Validation ----")
val_acc, _ = evaluate(clf, scaler, X_val, y_val)
print(f"Val accuracy: {val_acc*100:.2f}%")

---- Validation ----
Val accuracy: 79.17%


In [ ]:
print("---- Test ----")
test_acc, cm = evaluate(clf, scaler, X_test, y_test)
print(f"Test accuracy: {test_acc*100:.2f}%")
print(cm)

---- Test ----
Test accuracy: 71.88%
[[ 7  6  3  0  0  0]
 [ 1 14  0  1  0  0]
 [ 2  0 12  0  1  1]
 [ 0  0  1 13  2  0]
 [ 0  0  0  6  9  1]
 [ 0  1  0  1  0 14]]


In [ ]:
# 1. 맵 생성
segments_map = get_segments_map("00sequences.txt")

# 2. Vocabulary 학습용 서술자 추출
raw_descriptors = build_segmented_data("Data", segments_map, TRAIN_SUBJECTS, kmeans=None)
X_vocab = torch.cat(raw_descriptors).numpy()
# (이후 기존 kmeans fit 코드 실행)

# 3. 데이터셋 생성 (Histogram)
X_train, y_train = build_segmented_data("Data", segments_map, TRAIN_SUBJECTS, kmeans=kmeans, K=400)
X_test, y_test = build_segmented_data("Data", segments_map, TEST_SUBJECTS, kmeans=kmeans, K=400)

# 4. SVM 학습 및 평가 (기존 코드와 동일)
clf, scaler = train_svm(X_train, y_train)
acc, cm = evaluate(clf, scaler, X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")